# 2D Inversion

These are the steps to generate a 2D inversion from data collected at SAGE.

1. Load data
2. Extract a profile
3. Check strike direction
4. Interpolate onto same period map
5. Check data/edit
6. Setup inversion mesh
7. Run inversion
8. Check data fits
9. Change parameters and run again.


In [5]:
## Be sure to run this cell to enable Panel in VS Code
import panel as pn
pn.extension(comms='vscode')

## 1. Load data

We created an H5 file earlier with all the data.  We will load that into an `MTData` object so we can extract a profile.  

**NOTE**: Be sure to change the .h5 file path to your local path.

In [1]:
from mtpy import MTCollection

In [9]:
with MTCollection() as mc:
    mc.open_collection(r"c:\Users\jpeacock\OneDrive - DOI\SAGE\sage_2026.h5")
    md = mc.to_mt_data()
    md.utm_epsg = 32613 # set the UTM zone for the data

26:06:27T16:43:36 | INFO | line:1035 |mth5.mth5 | close_mth5 | Flushing and closing c:\Users\jpeacock\OneDrive - DOI\SAGE\sage_2026.h5


### Plot Station to get profile

Plot the station map to identify how you want to orient the profile.


In [10]:
station_plot = md.plot_stations()
station_plot.panel().servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'3e06970c-2bb9-48f2-ab45-9803551446f7': {'version…

The dominant strike is roughly NS, so profile lines should be roughly EW.

## 2. Extract Profile
From the map pick your profile lines.

In [18]:
end_point_01 = {"latitude": 35.95, "longitude": -106.2}
end_point_02 = {"latitude": 35.95, "longitude": -106.7}

In [19]:
# the syntax is (lon1, lat1, lon2, lat2, distance from profile in meters)
profile = md.get_profile(end_point_01["longitude"], end_point_01["latitude"], end_point_02["longitude"], end_point_02["latitude"], 1000)

In [20]:
profile

MTData(stations=11, surveys=7, lazy_stations=0, metadata_storage='cache', dataset_copy_mode='shallow', index_enabled=False)

In [21]:
profile_map = profile.plot_stations()
profile_map.panel().servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'0b53bb90-52ea-44f7-b682-ce35ffaee3bb': {'version…

## 3. Check Strike Direction

We want the profile to be roughly perpendicular to strike.

In [22]:
profile_strike = profile.plot_strike()
profile_strike.panel().servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'094c2ce6-f787-4735-9b55-b4888e74fd6c': {'version…

## 4. Interpolate and Check Data

We want the data to be uniform in that the transfer functions are mapped onto the same periods.  We will do that through interpolation. Then we will check the data quality so that we are trying to fit noise.

We want to interpolate over a range of periods and provide the inversion with enough data to constrain the model. But not so much that inversion is inefficient.  Play around with the period range and the number of periods


In [23]:
import numpy as np

In [25]:
# in log space 10^-3 to 10^3.3 with 32 points
interp_periods = np.logspace(-3, 3.3, 32)

In [26]:
profile_interp = profile.interpolate(interp_periods)

In [ ]:
#### Check the data